# Emma's Librarian: abordagem em plano (`--mode planned`) em vários modelos (Colab, GPU)

Roda os 72 casos de `core/harness/eval/cases.jsonl` no modo **planned**: o modelo (1) entrega um plano chamando
`submit_plan`, (2) executa cada tarefa com as ferramentas do assunto dela, (3) responde à pessoa. Mesmo código, prompt
(v1 das instruções, v3 do plano), seed 42 e nota (v2) das execuções anteriores, então o resultado se compara com as
colunas *roteado* e *completo* do Colab anterior (mesmo `index.jsonl`, se estiver na mesma pasta do Drive).

**O que mede e o que não mede.** Mede *eficácia* do plano. Os tempos daqui **não valem** para a RX 580.
No LFM3 local o plano gasta de 1150 a 2048 tokens de raciocínio antes de entregar; a pergunta deste notebook é se os
outros modelos (principalmente os que não raciocinam, como os Qwen com `enable_thinking: false`) planejam melhor.

Como usar:
1. Gere o zip do código (commit atual) e coloque na pasta `emmas-eval` do Drive, com o nome de `ZIP_NAME` abaixo:
   `git archive --format=zip -o emmas-harness-eval-v2.zip HEAD:emmas_librarian`
2. Ambiente de execução: GPU (L4 ou T4). Rode as células em ordem.
3. A célula 9 roda 4 casos de fumaça no primeiro modelo: confira que o plano sai antes de seguir para a bateria.
4. A execução é **retomável**: se a sessão cair, rode a célula 2 e as seguintes; o que já terminou é pulado.
5. Se o kernel reiniciar, rode a célula 2 de novo (ela define tudo o que as outras usam).

In [ ]:
# 1. A GPU
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout or 'SEM GPU: Ambiente de execução > Alterar tipo > GPU')
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout.strip().splitlines()[-1:])

In [ ]:
# 2. CONFIGURAÇÃO (o que quiser mudar fica aqui; rode de novo se o kernel reiniciar)
import os, json, subprocess, glob, time, datetime, hashlib
ZIP_NAME = 'emmas-harness-eval-v2.zip'
DRIVE_DIR = '/content/drive/MyDrive/emmas-eval'      # o zip fica aqui; os resultados vão para DRIVE_DIR/runs
APP = '/content/emmas'
# Os candidatos que passaram de 70% no Colab anterior, mais dois que ficaram abaixo, para ver se o plano os ajuda.
MODELS = ['lfm3-1.7b', 'qwen3.5-4b', 'qwen3.5-9b', 'gemma-4-e4b', 'qwen3.5-2b', 'lfm2.5-8b-a1b']
MODES = ['planned']
CASES = 'all'                    # 'all' ou ids separados por vírgula, ex.: 'art-2,proj-1'
SMOKE_CASES = 'proj-1,art-1,proj-5,ia-1'   # os 4 casos em que o LFM3 falhou/passou no teste local
LABEL = 'colab-plano'
LLAMA_REF = 'b12425'             # tag do llama.cpp. Se não compilar com a CUDA do Colab, troque por 'master': o commit real é gravado.
EXPECTED = None if CASES != 'all' else 72

# Modelos: id -> (repositório no Hugging Face, arquivo Q4_K_M). Os ids são os de modelCatalog.ts.
CATALOG = {
    'lfm3-1.7b': ('LiquidAI/LFM3-1.7B-GGUF', 'LFM3-1.7B-Q4_K_M.gguf'),
    'lfm2.5-8b-a1b': ('LiquidAI/LFM2.5-8B-A1B-GGUF', 'LFM2.5-8B-A1B-Q4_K_M.gguf'),
    'qwen3.5-2b': ('unsloth/Qwen3.5-2B-GGUF', 'Qwen3.5-2B-Q4_K_M.gguf'),
    'qwen3.5-4b': ('unsloth/Qwen3.5-4B-GGUF', 'Qwen3.5-4B-Q4_K_M.gguf'),
    'qwen3.5-9b': ('unsloth/Qwen3.5-9B-GGUF', 'Qwen3.5-9B-Q4_K_M.gguf'),
    'gemma-4-e4b': ('unsloth/gemma-4-E4B-it-GGUF', 'gemma-4-E4B-it-Q4_K_M.gguf'),
    'lfm2.5-1.2b-instruct': ('LiquidAI/LFM2.5-1.2B-Instruct-GGUF', 'LFM2.5-1.2B-Instruct-Q4_K_M.gguf'),
    'lfm2-1.2b-tool': ('LiquidAI/LFM2-1.2B-Tool-GGUF', 'LFM2-1.2B-Tool-Q4_K_M.gguf'),
    'lfm2.5-350m': ('LiquidAI/LFM2.5-350M-GGUF', 'LFM2.5-350M-Q4_K_M.gguf'),
}
PASS_THRESHOLD = 0.70
print('modelos:', MODELS, '| modos:', MODES, '| casos:', CASES)

In [ ]:
# 3. Drive e código do projeto
from google.colab import drive
import zipfile
drive.mount('/content/drive')
os.makedirs(f'{DRIVE_DIR}/runs', exist_ok=True)
zip_path = f'{DRIVE_DIR}/{ZIP_NAME}'
assert os.path.exists(zip_path), f'Falta {zip_path}: gere o zip (veja a célula de texto no topo) e coloque no Drive.'
if not os.path.exists(f'{APP}/package.json'):
    zipfile.ZipFile(zip_path).extractall(APP)
runner = open(f'{APP}/core/harness/eval/run/runEval.ts', encoding='utf-8').read()
assert "'planned'" in runner, 'O zip é antigo: não tem o modo planned. Gere o zip de novo do commit atual.'
print('código ok:', sorted(os.listdir(APP))[:10])

In [ ]:
# 4. Node 20 e as dependências (sem o Electron: --ignore-scripts; o --legacy-peer-deps respeita o package-lock)
!node --version 2>/dev/null | grep -q '^v2' || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1 && apt-get install -y nodejs >/dev/null 2>&1)
!node --version
%cd /content/emmas
!npm ci --ignore-scripts --legacy-peer-deps --no-audit --no-fund 2>&1 | tail -3
!npm rebuild better-sqlite3 2>&1 | tail -2

In [ ]:
# 5. Verificação: a biblioteca de teste F sobe em SQLite real neste sistema?
!cd /content/emmas && npx vitest run core/harness/__tests__/libraryF.test.ts core/harness/__tests__/plannedEpisode.test.ts 2>&1 | tail -8

In [ ]:
# 6. llama.cpp compilado aqui, na versão LLAMA_REF (o binário pronto não funcionou com a CUDA 13 do Colab)
!rm -rf /content/llama
!git clone --depth 1 --branch {LLAMA_REF} https://github.com/ggml-org/llama.cpp.git /content/llama
print('Compilando llama.cpp com CUDA (alguns minutos)...')
!cd /content/llama && cmake -B build -DGGML_CUDA=ON 2>&1 | tail -3 && cmake --build build --config Release -t llama-server -j $(nproc) 2>&1 | tail -3
SERVER = '/content/llama/build/bin/llama-server'
os.environ['LD_LIBRARY_PATH'] = '/content/llama/build/bin:/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
COMMIT = subprocess.run(['git', '-C', '/content/llama', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
LLAMA_BUILD = f'{LLAMA_REF}@{COMMIT}'          # o que vai para o registro de cada execução
listed = subprocess.run([SERVER, '--list-devices'], capture_output=True, text=True)
print(listed.stdout + listed.stderr)
import re
found = re.search(r'^\s*(CUDA\d+):', listed.stdout + listed.stderr, re.M)
assert found, 'llama.cpp NÃO enxerga a GPU e rodaria na CPU: confira a célula 1 e a saída acima.'
DEVICE = found.group(1)
print('build', LLAMA_BUILD, '| dispositivo', DEVICE)

In [ ]:
# 7. Modelos (direto do Hugging Face para o disco do Colab; o hash e o tamanho vão para o registro no Drive)
os.makedirs('/content/models', exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()

for model in MODELS:
    repo, name = CATALOG[model]
    dest = f'/content/models/{name}'
    if os.path.exists(dest):
        continue
    started = time.time()
    subprocess.run(['curl', '-L', '--fail', '-sS', '--retry', '3', '-o', dest + '.part', f'https://huggingface.co/{repo}/resolve/main/{name}'], check=True)
    os.rename(dest + '.part', dest)       # só existe com o nome final se o download terminou
    record = {'at': datetime.datetime.now().isoformat(timespec='seconds'), 'what': f'{repo}/{name}', 'bytes': os.path.getsize(dest),
              'seconds': round(time.time() - started, 1), 'sha256': sha256(dest)}
    open(f'{DRIVE_DIR}/runs/setup-colab.jsonl', 'a').write(json.dumps(record) + '\n')
    print(record['what'], record['bytes'] // 10**6, 'MB', record['seconds'], 's')

In [ ]:
# 8. A função que roda um modelo num modo (a mesma na fumaça e na bateria)
INDEX = f'{DRIVE_DIR}/runs/index.jsonl'

def already_done(model, mode):
    if not os.path.exists(INDEX):
        return False
    for line in open(INDEX):
        d = json.loads(line)
        if d.get('model') == model and d.get('mode') == mode and d.get('label') == f'{LABEL}-{mode}':
            return EXPECTED is None or d.get('cases') == EXPECTED
    return False

ENV = dict(os.environ, LLAMA_SERVER=SERVER, LLAMA_MODELS_DIR='/content/models', LLAMA_DEVICE=DEVICE,
           LLAMA_BACKEND='cuda', LLAMA_BUILD=LLAMA_BUILD, EVAL_RUNS_DIR=f'{DRIVE_DIR}/runs')

def run_eval(model, mode, cases, label):
    process = subprocess.Popen(['npx', 'vite-node', 'core/harness/eval/run/runEval.ts', '--', '--model', model, '--cases', cases,
                                '--mode', mode, '--label', label], cwd=APP, env=ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log = open(f'{DRIVE_DIR}/runs/log-{label}-{model}.txt', 'w')
    for line in process.stdout:
        log.write(line)
        if line.startswith(('PASS', 'FAIL')) or 'Error' in line:
            print(line.rstrip())
    process.wait()
    log.close()
    print('saída', process.returncode)

In [ ]:
# 9. Fumaça: 4 casos no primeiro modelo. O plano sai? Quantos tokens ele gasta? (rode antes da bateria)
run_eval(MODELS[0], 'planned', SMOKE_CASES, 'fumaca-plano')
last = sorted(glob.glob(f'{DRIVE_DIR}/runs/2026-*_{MODELS[0]}'))[-1]
for path in sorted(glob.glob(f'{last}/episodes/*.json')):
    e = json.load(open(path, encoding='utf-8'))
    r = e['record']
    first = r['steps'][0]['reply'] if r['steps'] else None
    print(e['id'], 'passou' if e['score']['pass'] else 'falhou', '| parou:', r['stopped'],
          '| plano:', r.get('plan'), '| tokens do plano:', first and first['timings']['generatedTokens'],
          '|', (r.get('error') or '')[:120])

In [ ]:
# 10. A bateria: cada modelo, cada modo. Retomável: o que já está no índice do Drive é pulado.
for mode in MODES:
    for model in MODELS:
        if already_done(model, mode):
            print('já feito:', model, mode)
            continue
        print(f'=== {model} / {mode}')
        run_eval(model, mode, CASES, f'{LABEL}-{mode}')

In [ ]:
# 11. Resultado: o plano contra os modos anteriores (roteado e completo, do Colab anterior, se estiver no mesmo índice)
import pandas as pd
rows = []
for line in open(INDEX):
    d = json.loads(line)
    label = str(d.get('label', ''))
    if d.get('cases') != 72 or not (label.startswith(LABEL) or label.startswith('colab-eficacia')):
        continue
    rows.append({'modelo': d['model'], 'abordagem': d['mode'], 'passou': d['passed'], 'taxa': round(d['passed'] / d['cases'], 2)})
table = pd.DataFrame(rows).drop_duplicates(['modelo', 'abordagem'], keep='last')
display(table.pivot(index='modelo', columns='abordagem', values='taxa'))
planned = table[table['abordagem'] == 'planned'].set_index('modelo')['taxa']
print('Com o plano, passam de %.0f%%:' % (PASS_THRESHOLD * 100), sorted(planned[planned >= PASS_THRESHOLD].index))

In [ ]:
# 12. O plano em detalhe: quantos planos saíram, quanto o modelo gastou para planejar, em que parou
stats = []
for config_path in sorted(glob.glob(f'{DRIVE_DIR}/runs/2026-*/config.json')):
    config = json.load(open(config_path, encoding='utf-8'))
    if not str(config.get('label') or '').startswith(LABEL) or config.get('mode') != 'planned':
        continue
    folder = os.path.dirname(config_path)
    plan_tokens, capped, invalid, empty, tasks = [], 0, 0, 0, []
    for path in glob.glob(f'{folder}/episodes/*.json'):
        record = json.load(open(path, encoding='utf-8'))['record']
        if record['steps']:
            reply = record['steps'][0]['reply']
            plan_tokens.append(reply['timings']['generatedTokens'])
            capped += reply['finishReason'] == 'length'
        invalid += 'ERR_INVALID_PLAN' in (record.get('error') or '')
        if record.get('plan') is not None:
            empty += len(record['plan']) == 0
            tasks.append(len(record['plan']))
    if plan_tokens:
        stats.append({'modelo': config['model']['id'], 'casos': len(plan_tokens), 'tokens do plano (média)': round(sum(plan_tokens) / len(plan_tokens)),
                      'bateu no limite': capped, 'plano inválido': invalid, 'plano vazio': empty,
                      'tarefas por plano (média)': round(sum(tasks) / max(len(tasks), 1), 2)})
display(pd.DataFrame(stats))

In [ ]:
# 13. Os casos que cada modelo errou no modo planned
for line in open(INDEX):
    d = json.loads(line)
    if str(d.get('label', '')).startswith(LABEL) and d.get('cases') == 72:
        print(d['model'], d['mode'], [r['id'] for r in d['results'] if not r['pass']])

In [ ]:
# 14. Baixar tudo o que foi gravado (já está no Drive; isto é só uma cópia)
import shutil
from google.colab import files
shutil.make_archive('/content/runs-colab-plano', 'zip', f'{DRIVE_DIR}/runs')
files.download('/content/runs-colab-plano.zip')